# Transformer Decoder 与文本生成

## 学习目标

使用 causal mask 训练一个极简 decoder-only Transformer，并实现逐 token 生成。

## 概念模型

训练时输入序列右移一位作为目标；causal mask 保证位置不能看到未来 token。推理时每次把新 token 拼接到上下文末尾。

In [ ]:
import torch
from torch import nn
torch.manual_seed(42)
vocab, length, width = 12, 6, 16
class TinyDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab, width)
        layer = nn.TransformerEncoderLayer(width, 2, batch_first=True)
        self.decoder = nn.TransformerEncoder(layer, 2)
        self.head = nn.Linear(width, vocab)
    def forward(self, tokens):
        n = tokens.size(1)
        h = self.embedding(tokens)
        mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
        return self.head(self.decoder(h, mask=mask))
model = TinyDecoder()
sequence = torch.randint(vocab, (8, length))
logits = model(sequence)
assert logits.shape == (8, length, vocab)
print(logits.shape)

### 实验 1：teacher forcing 训练

**实验目的**：输入序列去掉最后一个 token，目标去掉第一个 token，使每个位置预测下一个 token。logits 与 targets 展平后交给交叉熵。

causal mask 必须阻止位置看到未来 token，否则会发生标签泄漏。teacher forcing 训练看到真实历史，而生成时看到自身预测，两者存在 exposure bias。


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.03)
loss_fn = nn.CrossEntropyLoss()
for _ in range(3):
    optimizer.zero_grad(set_to_none=True)
    inputs, targets = sequence[:, :-1], sequence[:, 1:]
    loss = loss_fn(model(inputs).reshape(-1, vocab), targets.reshape(-1))
    loss.backward(); optimizer.step()
print('teacher-forcing loss:', loss.item())

### 实验 2：逐 token 自回归生成

**实验目的**：从 prefix 开始，每轮取最后位置 logits 的 argmax 作为下一个 token，并拼回输入。`eval()` 与 inference mode 保证确定性推理和低开销。

贪心解码不是唯一策略；采样、temperature、top-k/top-p 会改变多样性。还应处理 EOS、最大长度、位置编码上限和 KV cache。


In [ ]:
def generate(model, prefix, steps):
    model.eval(); result = prefix.clone()
    with torch.inference_mode():
        for _ in range(steps):
            next_token = model(result)[:, -1].argmax(dim=-1, keepdim=True)
            result = torch.cat([result, next_token], dim=1)
    return result
generated = generate(model, sequence[:1, :2], 3)
print('generated:', generated.tolist())
assert generated.shape == (1, 5)

## 官方教程补充

**对应官方源文件：** `intermediate_source/seq2seq_translation_tutorial.py`、`beginner_source/chatbot_tutorial.py`、`intermediate_source/transformer_building_blocks.py`

官方 seq2seq 教程区分训练与生成：训练可用 teacher forcing 并行处理已知目标，推理只能根据已生成 token 自回归前进。decoder 必须使用 causal mask，padding 仍需单独屏蔽；生成在 EOS 或最大长度停止。贪心、采样和 beam search 改变质量/多样性/计算权衡，缓存 K/V 可避免每步重复计算历史。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 实验 3：KV cache——增量 attention 的关键优化

在上面的 `generate` 中，每生成一个 token，`model(result)` 都会重新处理整个 prefix。历史 token 的 Key/Value 并没有改变，却被重复计算。KV cache 的思路是：第一次处理 prefix 时保存历史 K/V，之后每一步只为新 token 计算 query、key、value，再把新的 K/V 追加到 cache。

这里使用最小的单头 causal attention，专门展示 cache 的数据结构；现有 `nn.TransformerEncoder` 接口本身不会自动暴露 KV cache。

In [ ]:
import math

class TinySelfAttention(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.q_proj = nn.Linear(width, width, bias=False)
        self.k_proj = nn.Linear(width, width, bias=False)
        self.v_proj = nn.Linear(width, width, bias=False)

    def full(self, h):
        q, k, v = self.q_proj(h), self.k_proj(h), self.v_proj(h)
        scores = q @ k.transpose(-2, -1) / math.sqrt(h.size(-1))
        causal = torch.triu(torch.ones(h.size(1), h.size(1), dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal, float('-inf'))
        return scores.softmax(dim=-1) @ v

    def step(self, h_new, cache=None):
        q = self.q_proj(h_new)
        k_new, v_new = self.k_proj(h_new), self.v_proj(h_new)
        keys, values = (k_new, v_new) if cache is None else (torch.cat([cache[0], k_new], 1), torch.cat([cache[1], v_new], 1))
        scores = q @ keys.transpose(-2, -1) / math.sqrt(h_new.size(-1))
        return scores.softmax(dim=-1) @ values, (keys, values)

attention = TinySelfAttention(width)
hidden = torch.randn(1, 5, width)
full_output = attention.full(hidden)
cache, incremental = None, []
for t in range(hidden.size(1)):
    output, cache = attention.step(hidden[:, t:t+1], cache)
    incremental.append(output)
incremental = torch.cat(incremental, dim=1)
torch.testing.assert_close(full_output, incremental, rtol=1e-5, atol=1e-5)
print('cache K shape:', cache[0].shape, 'cache V shape:', cache[1].shape)
print('max error:', (full_output - incremental).abs().max().item())

### 读懂 KV cache 的收益与代价

第 `t` 步 cache 的 K/V 形状是 `(batch, t, width)`；新 query 只有 `(batch, 1, width)`，因此 attention score 是 `(batch, 1, t)`。无 cache 时会反复计算长度为 1、2、…、T 的完整序列；有 cache 时历史 K/V 只计算一次，但新 query 仍需与全部历史 K 做 attention。

cache 用显存换计算：每层通常保存 K 和 V，大小随 batch、层数、头数、上下文长度和 dtype 增长。实际 decoder 还要处理多头维度、位置编码、padding、最大上下文长度和不同请求的 cache 生命周期。

## 检查点

解释 teacher forcing 与生成时输入的区别，说明 causal mask 的上三角为什么必须屏蔽。

## 试一试

将 argmax 改成 temperature sampling，并限制最大上下文长度。

## 常见错误与调试

目标没有右移、训练时看到未来 token、mask device 不一致、把序列维和 batch 维混淆、生成时忘记只取最后位置。